# Weather Data

In [ ]:
import cdsapi # important in order to use the CDS API
import pandas as pd
import numpy as np
import zipfile
import os
from glob import glob
import holidays

## CDS API Call
To make use of the CDSAPI and access the ERA5-Land dataset via the C3S data store, the following steps are mandatory.

### 1. Account & Credentials
Register for a free account on the Climate Data Store (CDS) website (cds.climate.copernicus.eu) or log in.
You can find your Personal Access Token in your profile (https://cds.climate.copernicus.eu/profile).

### 2. Create a .cdsapirc file
In the home directory ($HOME/.cdsapirc on Linux/Mac, or C:\Users\<Name>\.cdsapirc on Windows), create a file with the following content:

```
url: https://cds.climate.copernicus.eu/api
key: <YOUR-PERSONAL-ACCESS-TOKEN>
```
The cdsapi client automatically reads the token and URL from this file – therefore, in the code, cdsapi.Client() without any parameters is sufficient.

### 3. Install a Python package

```
bash
pip install "cdsapi>=0.7.7"
```
Older versions no longer work reliably with the new CDS system (since the 2024 migration), so please use the latest version.

### 4. Accept the licence terms
For each dataset (in this case, ERA5 Land hourly time-series data from 1950 to present), you must manually accept the Terms of Use on the website once. This can be done via the dataset’s download page, at the bottom of the form. Without this, the API request will fail, even if you have the correct key.


In [ ]:
# automatically read the last date from the taxi data set as end date for the weather data

taxi_data_processed = pd.read_parquet('../data/processed/taxi_data_processed.parquet')

end_date = pd.to_datetime(
    taxi_data_processed["Trip End Timestamp"].max(),
    format="%m/%d/%Y %I:%M:%S %p"
).strftime("%Y-%m-%d")

print(end_date)

In [ ]:
# config dictionary for the CDS API call

CONFIG = {
    # select the variables we are interested in; find the names of the variables at https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview or make use of the web based dataset picker 
    # on https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=download 
    "variables": [
        "2m_temperature", 
        "total_precipitation",
        "snow_cover",
        "snow_depth",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind"
    ],
    "date": f"2024-01-01/{end_date}",
    "dir_name": "era5_data.zip",
}

In [ ]:
# API call of the CDS API which can also be generated on https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=download

dataset = "reanalysis-era5-land-timeseries"

# use the values from the config dictionary 
request = {
    "variable": CONFIG["variables"],
    "location": {"longitude": -87.8, "latitude": 42}, # coordinates of Chicago
    "date": CONFIG["date"],
    "data_format": "csv"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download(f"../data/{CONFIG['dir_name']}")

2026-06-22 16:14:05,177 INFO [2026-02-16T00:00:00] - To generate this ERA5-land hourly time series dataset, **homogenisation conventions have been applied to the ERA5 source GRIB data** to ensure consistency, usability, and alignment across chosen variables and time steps. The processed data were then written to an **ARCO Zarr archive**, enabling efficient cloud-optimised access and scalable data retrieval. Please refer to the [user guide](https://confluence.ecmwf.int/x/Dg32Hw) for details.

- The dataset presented here is a subset of selected parameters from the full [CDS ERA5 hourly data on single levels (1940–present)](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land?tab=overview). **Requirements for additional parameters may be considered**. Please raise your request with ECMWF Support [here](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202).
2026-06-22 16:14:05,178 INFO Request ID is 07f6f7f8-a4cc-456c-825a-7066364751f7
2026-06-22 16:14:05,257 INFO st

ab20def0ad363ec3d4e18bd3617825da.zip:   0%|          | 0.00/652k [00:00<?, ?B/s]

'era5_data.zip'

## Load individual CSV files 

In [ ]:
# print what files are present in the zip
with zipfile.ZipFile(f"../data/{CONFIG['dir_name']}", "r") as zip_ref:
    zip_ref.extractall("era5_data")

print(os.listdir("era5_data"))

['reanalysis-era5-land-timeseries-sfc-2m-temperature26j6g913.csv', 'reanalysis-era5-land-timeseries-sfc-pressure-precipitationl9x_8x7v.csv', 'reanalysis-era5-land-timeseries-sfc-snowtxoqj74o.csv', 'reanalysis-era5-land-timeseries-sfc-windxt9xvkko.csv']


In [ ]:
# list all CSV files
files = glob("era5_data/*.csv")

# dictionary for the dataframes
dfs = {}

for file in files:
    filename = os.path.basename(file)

    # find the correct csv based on the file names
    if "temperature" in filename:
        key = "temp"
    elif "precipitation" in filename:
        key = "precip"
    elif "snow" in filename:
        key = "snow"
    elif "wind" in filename:
        key = "wind"

    # load individual csv files 
    dfs[key] = pd.read_csv(file, encoding="latin1")

    print(f"\n--- {key} ---")
    print(dfs[key].head())


--- temp ---
            valid_time        t2m  latitude  longitude
0  2024-01-01 00:00:00  274.39615      42.0      -87.8
1  2024-01-01 01:00:00  274.24170      42.0      -87.8
2  2024-01-01 02:00:00  274.05762      42.0      -87.8
3  2024-01-01 03:00:00  273.90906      42.0      -87.8
4  2024-01-01 04:00:00  274.01672      42.0      -87.8

--- precip ---
            valid_time        tp  latitude  longitude
0  2024-01-01 00:00:00  0.000281      42.0      -87.8
1  2024-01-01 01:00:00  0.000150      42.0      -87.8
2  2024-01-01 02:00:00  0.000030      42.0      -87.8
3  2024-01-01 03:00:00  0.000014      42.0      -87.8
4  2024-01-01 04:00:00  0.000037      42.0      -87.8

--- snow ---
            valid_time     snowc       sde  latitude  longitude
0  2024-01-01 00:00:00  6.929688  0.007812      42.0      -87.8
1  2024-01-01 01:00:00  8.179688  0.008789      42.0      -87.8
2  2024-01-01 02:00:00  8.804688  0.008789      42.0      -87.8
3  2024-01-01 03:00:00  8.873047  0.008789    

## Merge individual Dataframes

In [ ]:
df_merged = dfs["temp"].copy()

df_merged = df_merged.merge(
    dfs["precip"],
    on=["valid_time", "latitude", "longitude"],
    how="left",
    suffixes=("", "_snow")
)
df_merged = df_merged.merge(
    dfs["snow"],
    on=["valid_time", "latitude", "longitude"],
    how="left",
    suffixes=("", "_snow")
)
df_merged = df_merged.merge(
    dfs["wind"],
    on=["valid_time", "latitude", "longitude"],
    how="left",
    suffixes=("", "_snow")
)

df_merged

,valid_time,t2m,latitude,longitude,tp,snowc,sde,u10,v10
0,2024-01-01 00:00:00,274.39615,42.0,-87.8,0.000281,6.929688,7.812500e-03,2.705109,-5.971687
1,2024-01-01 01:00:00,274.24170,42.0,-87.8,0.000150,8.179688,8.789062e-03,2.658508,-7.036841
2,2024-01-01 02:00:00,274.05762,42.0,-87.8,0.000030,8.804688,8.789062e-03,2.478607,-7.009237
3,2024-01-01 03:00:00,273.90906,42.0,-87.8,0.000014,8.873047,8.789062e-03,2.442825,-6.861008
4,2024-01-01 04:00:00,274.01672,42.0,-87.8,0.000037,8.890625,8.789062e-03,2.084305,-6.889282
...,...,...,...,...,...,...,...,...,...
20707,2026-05-12 19:00:00,293.30435,42.0,-87.8,0.000342,0.000000,-7.345365e-24,1.670914,4.648880
20708,2026-05-12 20:00:00,293.64343,42.0,-87.8,0.000003,0.000000,-7.345365e-24,1.859116,3.183975
20709,2026-05-12 21:00:00,293.37470,42.0,-87.8,0.000053,0.000000,-7.345365e-24,2.806747,3.596573
20710,2026-05-12 22:00:00,292.62510,42.0,-87.8,0.000749,0.000000,-7.345365e-24,3.238815,4.192703


## Data Preprocessing

In [ ]:
# check for missing values
print("--- Missing values ---")
print(df_merged.isna().sum())
print("")

# check for data types
print("--- Data types ---")
print(df_merged.dtypes)
print("")

# short statistical description of the data
print("--- Stat description ---")
print(df_merged.describe())

--- Missing values ---
valid_time    0
t2m           0
latitude      0
longitude     0
tp            0
snowc         0
sde           0
u10           0
v10           0
dtype: int64
--- Data types ---
valid_time     object
t2m           float64
latitude      float64
longitude     float64
tp            float64
snowc         float64
sde           float64
u10           float64
v10           float64
dtype: object
--- Stat description ---
                t2m      latitude     longitude            tp         snowc  \
count  20712.000000  2.071200e+04  2.071200e+04  2.071200e+04  20712.000000   
mean     283.151036  4.200000e+01 -8.780000e+01  1.109412e-04     10.311244   
std       10.784377  1.136896e-12  2.651810e-11  5.408161e-04     25.109695   
min      248.497120  4.200000e+01 -8.780000e+01 -3.736932e-08      0.000000   
25%      274.883240  4.200000e+01 -8.780000e+01  0.000000e+00      0.000000   
50%      283.333980  4.200000e+01 -8.780000e+01  0.000000e+00      0.000000   
75%      29

### Clean up

In [ ]:
# drop unnecessary columns as always the same coordinates for all data points
df_merged = df_merged.drop(['latitude', 'longitude'], axis=1)

# rename columns to improve readability and interpretabilty
df_merged = df_merged.rename(
    columns={
        'valid_time': 'time_step',
        't2m': '2m_temp',
        'tp': 'total_precip',
        'snowc': 'snow_cov',
        'sde': 'snow_depth'
    }
)
df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,7.812500e-03,2.705109,-5.971687
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,8.789062e-03,2.658508,-7.036841
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,8.789062e-03,2.478607,-7.009237
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,8.789062e-03,2.442825,-6.861008
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,8.789062e-03,2.084305,-6.889282
...,...,...,...,...,...,...,...
20707,2026-05-12 19:00:00,293.30435,0.000342,0.000000,-7.345365e-24,1.670914,4.648880
20708,2026-05-12 20:00:00,293.64343,0.000003,0.000000,-7.345365e-24,1.859116,3.183975
20709,2026-05-12 21:00:00,293.37470,0.000053,0.000000,-7.345365e-24,2.806747,3.596573
20710,2026-05-12 22:00:00,292.62510,0.000749,0.000000,-7.345365e-24,3.238815,4.192703


In [ ]:
# change dtype to datetime object for column time_step
df_merged['time_step'] = pd.to_datetime(df_merged['time_step'])

# set lower boundary for precipitation and snow depth to get rid of physically impossible values, e.g. negative preciptation (porbably due to accumulation of small floating point operation errors)
df_merged['total_precip'] = df_merged['total_precip'].clip(lower=0)
df_merged['snow_depth'] = df_merged['snow_depth'].clip(lower=0)

df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282
...,...,...,...,...,...,...,...
20707,2026-05-12 19:00:00,293.30435,0.000342,0.000000,0.000000,1.670914,4.648880
20708,2026-05-12 20:00:00,293.64343,0.000003,0.000000,0.000000,1.859116,3.183975
20709,2026-05-12 21:00:00,293.37470,0.000053,0.000000,0.000000,2.806747,3.596573
20710,2026-05-12 22:00:00,292.62510,0.000749,0.000000,0.000000,3.238815,4.192703


### Conversion of units

In [ ]:
# convert temperature from Kelvin to Celsius via formula C = K - 273.15
df_merged['2m_temp_c'] = df_merged['2m_temp'] - 273.15

# convert from m to mm (common unit for precipitation)
df_merged['total_precip_mm'] = df_merged['total_precip'] * 1000

df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10,2m_temp_c,total_precip_mm
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687,1.24615,0.280723
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841,1.09170,0.150489
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237,0.90762,0.030188
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008,0.75906,0.013527
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282,0.86672,0.037491
...,...,...,...,...,...,...,...,...,...
20707,2026-05-12 19:00:00,293.30435,0.000342,0.000000,0.000000,1.670914,4.648880,20.15435,0.341564
20708,2026-05-12 20:00:00,293.64343,0.000003,0.000000,0.000000,1.859116,3.183975,20.49343,0.002533
20709,2026-05-12 21:00:00,293.37470,0.000053,0.000000,0.000000,2.806747,3.596573,20.22470,0.052869
20710,2026-05-12 22:00:00,292.62510,0.000749,0.000000,0.000000,3.238815,4.192703,19.47510,0.748754


## Data Export
Export the data into its own parquet file in order to merge it with POI and taxi data later on.

In [ ]:
# Export weather data table to use in other notebooks
df_merged.to_parquet("../data/processed/weather_data_processed.parquet")